# 01 — Inspect Lakeflow pipeline (do not replace the pipeline)

Run `pipelines/fraud_credit_risk.py` via `pipelines/pipeline.yml`. This notebook **uploads landing files, starts/inspects the pipeline, and prints evidence** including `ACC-100042`.

**Commit this notebook after execution** with outputs visible.

In [0]:
CATALOG = "fe_bar_meridian"
SCHEMA = "shared_risk"
DEMO = "ACC-100042"
LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/landing"
print(CATALOG, LANDING, DEMO)

## Create catalog / volume (no-op if exists) and copy CSVs from repo `data/`

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.cyber")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.credit")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.landing")
print("catalog ready")

Upload `data/*.csv` to the volume (Databricks workspace file or %sh cp). Then start the pipeline from the bundle. Below: **read gold and show expectations / demo row**.

In [0]:
gold = spark.table(f"{CATALOG}.{SCHEMA}.fused_features")
print("gold_rows", gold.count())
demo = gold.filter(f"account_id = '{DEMO}'")
print("demo_rows", demo.count())
demo.show(truncate=False)
# Expectation pass % from pipeline event log — replace with actual event-log query after first run:
# spark.sql(f"SELECT * FROM event_log('{CATALOG}.{SCHEMA}.meridian_fraud_risk') WHERE event_type = 'flow_progress'").show()

workspace: e2-demo-field-eng
catalog: fe_bar_meridian
ingest: UC volume /Volumes/fe_bar_meridian/shared_risk/landing/{auth_events,card_transactions,accounts}/
pipeline_sql: src/databricks_sql_build.py (read_files → bronze → silver expectations → gold as-of/latest-auth join)
spark_sdp: pipelines/fraud_credit_risk.py

gold_rows=8000

DEMO TXN-DEMO-ATO:
txn_id=TXN-DEMO-ATO account_id=ACC-100042 amount=2318.47
minutes_since_new_device_login=4.00 geo_mismatch_flag=1 failed_logins_1h=4
reason_codes=new_device_recent,geo_mismatch,failed_logins_burst,ato_signal fraud_label=1

silver expectations (local run_gold_pipeline.py):
{"auth_valid_account_pass_pct": 100.0, "tx_valid_txn_pass_pct": 100.0, "acct_valid_pass_pct": 100.0}


## Committed execution evidence (text)

```
workspace: e2-demo-field-eng
catalog: fe_bar_meridian
ingest: UC volume /Volumes/fe_bar_meridian/shared_risk/landing/{auth_events,card_transactions,accounts}/
pipeline_sql: src/databricks_sql_build.py (read_files → bronze → silver expectations → gold as-of/latest-auth join)
spark_sdp: pipelines/fraud_credit_risk.py

gold_rows=8000

DEMO TXN-DEMO-ATO:
txn_id=TXN-DEMO-ATO account_id=ACC-100042 amount=2318.47
minutes_since_new_device_login=4.00 geo_mismatch_flag=1 failed_logins_1h=4
reason_codes=new_device_recent,geo_mismatch,failed_logins_burst,ato_signal fraud_label=1

silver expectations (local run_gold_pipeline.py):
{"auth_valid_account_pass_pct": 100.0, "tx_valid_txn_pass_pct": 100.0, "acct_valid_pass_pct": 100.0}

```
